# 🌐 Khám Phá Toàn Văn Dữ Liệu Gốc Theo Từng Domain
### ViBioMIR / Road to AI 2026 — Raw Medical Corpus Inspector

> **Mục đích:**
> - Xem lại toàn bộ **văn bản nguyên gốc chưa tách chunk** (1.025.647 bài viết) theo từng domain nguồn.
> - Không dùng widget phức tạp, hiển thị trực quan, sạch sẽ và tự động ngắt dòng (`white-space: pre-wrap`) không bị tràn ngang màn hình.

In [ ]:
# 1. Khởi tạo môi trường & Cấu hình tự động ngắt dòng văn bản
import os
import sys
from pathlib import Path
import pandas as pd
from IPython.display import HTML, display

# Đặt Current Working Directory về thư mục gốc dự án
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "src").exists() else CURRENT_DIR.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import các hàm đọc dữ liệu từ src
from src.retrieval.doc_reader import (
    get_domains_catalog,
    get_documents_by_domain,
    get_random_doc_by_domain,
    lookup_raw_doc_by_id,
    display_raw_document,
)
from src.ingestion.cleaner import normalize_text

# Nhúng CSS ngắt dòng tự động
display(HTML("""
<style>
    .output_text pre, .output_subarea pre, div.output_area pre {
        white-space: pre-wrap !important;
        word-wrap: break-word !important;
        word-break: break-word !important;
        font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif !important;
    }
</style>
"""))
print(f"✅ Đã sẵn sàng! Thư mục làm việc: {PROJECT_ROOT.resolve()}")

--- 
## 📊 1. Thống Kê Danh Mục Domain Trong Kho Dữ Liệu

Xem danh sách các domain có nhiều bài viết nhất (tổng 1.025.647 bài viết):

In [ ]:
catalog_df = get_domains_catalog()

print(f"Tổng số domain: {catalog_df['domain'].nunique()} | Tổng số bài viết: {catalog_df['article_count'].sum():,}")

# Hiển thị Top 25 domain lớn nhất
summary = catalog_df.copy()
summary["avg_char_length"] = summary["avg_char_length"].round(0).astype(int)
summary.columns = ["Domain", "Ngôn ngữ", "Số bài viết", "Độ dài ký tự TB", "Tiêu đề mẫu"]
summary.head(25).reset_index(drop=True)

--- 
## 📖 2. Xem Toàn Văn Bài Viết Của Bất Kỳ Domain Nào

Chỉ cần thay đổi biến `DOMAIN` (ví dụ: `suckhoedoisong.vn`, `www.vinmec.com`, `thanhnien.vn`, `www.120ask.com`...) và bấm chạy:

In [ ]:
# === ĐỔI TÊN DOMAIN BẠN MUỐN XEM Ở ĐÂY ===
DOMAIN = "suckhoedoisong.vn"  # Ví dụ: "www.vinmec.com", "thanhnien.vn", "www.120ask.com"
SO_LUONG_BAI = 2             # Số bài viết muốn hiển thị
TU_KHOA_LOC = ""             # Có thể nhập từ khóa lọc (ví dụ: "dạ dày", "sốt", "tiểu cầu") hoặc để trống ""

docs = get_documents_by_domain(
    domain=DOMAIN,
    limit=SO_LUONG_BAI,
    keyword_filter=TU_KHOA_LOC,
)

if docs:
    for i, d in enumerate(docs, 1):
        display(HTML(f"<h4 style='color:#1d4ed8; margin: 16px 0 4px 0;'>Bài viết #{i} từ domain: <code>{DOMAIN}</code></h4>"))
        display_raw_document(d)
else:
    print(f"❌ Không tìm thấy bài viết nào trong domain '{DOMAIN}' khớp với từ khóa '{TU_KHOA_LOC}'!")

--- 
## 🎲 3. Lấy Ngẫu Nhiên 1 Bài Viết Từ Domain Bất Kỳ

Mỗi lần chạy cell này sẽ bốc ngẫu nhiên một bài viết nguyên gốc từ domain chỉ định:

In [ ]:
# Chọn domain để bốc ngẫu nhiên bài viết
RANDOM_DOMAIN = "www.vinmec.com"

rand_doc = get_random_doc_by_domain(RANDOM_DOMAIN)
if rand_doc:
    display_raw_document(rand_doc)
else:
    print(f"Không tìm thấy dữ liệu cho domain: {RANDOM_DOMAIN}")

--- 
## 🔍 4. Tra Cứu Toàn Văn Gốc Theo Doc ID Bất Kỳ

Nhập ID bài viết bất kỳ (ví dụ: `733125`, `545347`, `760025`) để xem lại toàn bộ văn bản gốc:

In [ ]:
# Nhập Doc ID bạn muốn tra cứu
DOC_ID = 733125

doc_result = lookup_raw_doc_by_id(DOC_ID)
if doc_result:
    display_raw_document(doc_result)
else:
    print(f"❌ Không tìm thấy Doc ID {DOC_ID} trong kho dữ liệu!")

--- 
## 🧹 5. Kiểm Tra Rác Chân Trang Của Domain (Hotline / Copyright / Ads)

So sánh 200 ký tự cuối của bài viết trước và sau khi chạy qua bộ làm sạch boilerplate:

In [ ]:
# Chọn domain cần kiểm tra mức độ sạch
AUDIT_DOMAIN = "suckhoedoisong.vn"

sample_list = get_documents_by_domain(AUDIT_DOMAIN, limit=1)
if sample_list:
    raw_item = sample_list[0]
    raw_text = raw_item["text"]
    clean_text = normalize_text(raw_text, strip_boilerplate=True)
    
    chars_diff = len(raw_text) - len(clean_text)
    
    display(HTML(f"""
    <div style='border:1px solid #cbd5e1; border-radius:8px; padding:16px; margin:12px 0; background:#ffffff;'>
        <h4 style='color:#1d4ed8; margin-top:0;'>Kiểm tra mức độ sạch: <code>{AUDIT_DOMAIN}</code> (Doc #{raw_item['doc_id']})</h4>
        <div style='font-size:13px; color:#475569; margin-bottom:10px;'>
            Độ dài gốc: <b>{len(raw_text):,}</b> ký tự | Sau làm sạch: <b>{len(clean_text):,}</b> ký tự | Đã loại bỏ: <b style='color:#dc2626;'>{chars_diff} ký tự rác</b>
        </div>
        <div style='display:grid; grid-template-columns: 1fr 1fr; gap:14px;'>
            <div style='background:#fef2f2; border:1px solid #fecaca; padding:10px; border-radius:6px; font-size:13px;'>
                <b style='color:#b91c1c;'>200 ký tự cuối GỐC (chứa rác/hotline/bản quyền):</b><br>
                <pre style='margin-top:6px; white-space:pre-wrap;'>{raw_text[-200:]}</pre>
            </div>
            <div style='background:#f0fdf4; border:1px solid #bbf7d0; padding:10px; border-radius:6px; font-size:13px;'>
                <b style='color:#15803d;'>200 ký tự cuối SAU LÀM SẠCH (đã cắt rác):</b><br>
                <pre style='margin-top:6px; white-space:pre-wrap;'>{clean_text[-200:]}</pre>
            </div>
        </div>
    </div>
    """))